# Proyecto Integrador — Semana 02  
# 04 Gold Perfil de Usuario — Daniel Guzmán

## Objetivo

Construir una tabla Gold para identificar usuarios con comportamiento de riesgo a partir de sus transacciones históricas.

## Entrada

- `workspace.silver.transactions_daniel`

## Salida Gold

- `workspace.gold.perfil_usuario_daniel`

## Criterio de riesgo

La categoría de riesgo se calculará con base en la tasa de fraude personal y la cantidad de fraudes:

- **Alta:** usuarios con al menos 3 fraudes o tasa de fraude mayor o igual a 1%.
- **Media:** usuarios con al menos 1 fraude o tasa de fraude mayor o igual a 0.3%.
- **Baja:** usuarios sin señales relevantes de fraude.

Este criterio permite priorizar usuarios con mayor frecuencia o proporción de fraude.

In [0]:
from pyspark.sql import functions as F

MI_NOMBRE = "daniel"
CATALOG = "workspace"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

df_silver = spark.table(f"{CATALOG}.silver.transactions_{MI_NOMBRE}")

print(f"Filas Silver: {df_silver.count():,}")
print(f"Columnas Silver: {len(df_silver.columns)}")

display(df_silver.limit(5))

In [0]:
# Gold — Perfil de usuario en riesgo

df_perfil_usuario = (
    df_silver
    .groupBy("user_id")
    .agg(
        F.count("transaction_id").alias("total_transacciones"),
        F.sum(F.when(F.col("is_fraud") == 1, 1).otherwise(0)).alias("total_fraudes"),
        F.sum(F.when(F.col("is_fraud") == 0, 1).otherwise(0)).alias("total_legitimas"),
        F.sum(F.when(F.col("is_fraud").isNull(), 1).otherwise(0)).alias("sin_label"),
        F.sum(F.when(F.col("is_fraud").isin(0, 1), 1).otherwise(0)).alias("transacciones_etiquetadas"),
        F.round(F.sum("amount_abs"), 2).alias("monto_total"),
        F.round(
            F.sum(F.when(F.col("is_fraud") == 1, F.col("amount_abs")).otherwise(0)),
            2
        ).alias("monto_fraudulento"),
        F.countDistinct("card_id").alias("num_tarjetas"),
        F.countDistinct("mcc").alias("num_categorias_mcc"),
        F.min("transaction_date").alias("primer_transaccion"),
        F.max("transaction_date").alias("ultima_transaccion")
    )
    .withColumn(
        "tasa_fraude_pct",
        F.when(
            F.col("transacciones_etiquetadas") > 0,
            F.round(F.col("total_fraudes") / F.col("transacciones_etiquetadas") * 100, 4)
        ).otherwise(None)
    )
    .withColumn(
        "dias_activo",
        F.datediff(F.col("ultima_transaccion"), F.col("primer_transaccion"))
    )
    .withColumn(
        "categoria_riesgo",
        F.when(
            (F.col("total_fraudes") >= 3) | (F.col("tasa_fraude_pct") >= 1),
            "Alta"
        )
        .when(
            (F.col("total_fraudes") >= 1) | (F.col("tasa_fraude_pct") >= 0.3),
            "Media"
        )
        .otherwise("Baja")
    )
)

df_perfil_usuario = df_perfil_usuario.select(
    "user_id",
    "total_transacciones",
    "total_fraudes",
    "tasa_fraude_pct",
    "monto_total",
    "monto_fraudulento",
    "num_tarjetas",
    "num_categorias_mcc",
    "primer_transaccion",
    "ultima_transaccion",
    "dias_activo",
    "categoria_riesgo",
    "total_legitimas",
    "sin_label",
    "transacciones_etiquetadas"
)

display(
    df_perfil_usuario
    .orderBy(F.col("categoria_riesgo"), F.col("tasa_fraude_pct").desc(), F.col("total_fraudes").desc())
    .limit(20)
)

In [0]:
df_perfil_usuario.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.gold.perfil_usuario_{MI_NOMBRE}")

print(f"Tabla guardada: {CATALOG}.gold.perfil_usuario_{MI_NOMBRE}")
print(f"Usuarios perfilados: {df_perfil_usuario.count():,}")

In [0]:
print("Distribución de usuarios por categoría de riesgo:")

df_distribucion_riesgo = (
    df_perfil_usuario
    .groupBy("categoria_riesgo")
    .agg(
        F.count("*").alias("total_usuarios"),
        F.round(F.avg("tasa_fraude_pct"), 4).alias("tasa_promedio_fraude"),
        F.sum("total_fraudes").alias("fraudes_totales"),
        F.round(F.sum("monto_fraudulento"), 2).alias("monto_fraudulento_total")
    )
    .orderBy(
        F.when(F.col("categoria_riesgo") == "Alta", 1)
         .when(F.col("categoria_riesgo") == "Media", 2)
         .otherwise(3)
    )
)

display(df_distribucion_riesgo)

## Resultados — Perfil de usuario en riesgo

La mayoría de los usuarios quedaron clasificados en riesgo **Alta**, con **1,034 usuarios**. Este grupo concentra **13,092 fraudes** y un monto fraudulento total de **1,719,668.29**.

El grupo de riesgo **Media** contiene **162 usuarios**, con **240 fraudes** y un monto fraudulento total de **29,336.49**.

El grupo de riesgo **Baja** contiene **23 usuarios** y no registra fraudes.

Esto muestra que el criterio de riesgo separa claramente a los usuarios con mayor historial fraudulento, concentrando casi todo el fraude en la categoría Alta.

In [0]:
print("Top usuarios de mayor riesgo:")

display(
    df_perfil_usuario
    .filter(F.col("categoria_riesgo").isin("Alta", "Media"))
    .orderBy(
        F.col("total_fraudes").desc(),
        F.col("tasa_fraude_pct").desc(),
        F.col("monto_fraudulento").desc()
    )
    .limit(20)
)

## Documentación Gold — Perfil de Usuario

Se creó la tabla `workspace.gold.perfil_usuario_daniel` para identificar usuarios con comportamiento potencialmente riesgoso.

### Métricas calculadas

Por cada usuario se calcularon:

- Total de transacciones.
- Total de fraudes.
- Tasa de fraude personal.
- Monto total transaccionado.
- Monto fraudulento.
- Número de tarjetas distintas usadas.
- Número de categorías MCC distintas.
- Primera y última transacción.
- Días activo.
- Categoría de riesgo.

### Criterio de categoría de riesgo

La categoría `categoria_riesgo` se asignó así:

- **Alta:** usuarios con al menos 3 fraudes o tasa de fraude mayor o igual a 1%.
- **Media:** usuarios con al menos 1 fraude o tasa de fraude mayor o igual a 0.3%.
- **Baja:** usuarios sin señales relevantes de fraude.

### Decisión metodológica

La tasa de fraude personal se calculó usando únicamente transacciones etiquetadas.  
Los registros sin etiqueta se conservaron como parte del historial del usuario, pero no se usaron como denominador para la tasa de fraude.
